# Import Libraries

In [ ]:
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver  
from langgraph.store.postgres import PostgresStore 
from langchain.tools import tool,ToolRuntime
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain_groq import ChatGroq

import os 
DB_URI = os.getenv("DATABASE_URL")
# DB_URI

# Tools

In [53]:
@dataclass
class UserContext:
    user_id: str

@tool
def save_user_memory(name: str,language:str,runtime: ToolRuntime[UserContext])-> str:
    "Save user name and preferred programming language to memory"
    runtime.store.put(
        ("users",),
        runtime.context.user_id,
        {"name":name,"language":language}
    )

    return "Saved user memory successfully"

@tool
def get_user_memory(runtime: ToolRuntime[UserContext])-> str:
    "Get user name and preferred programming language from memory"
    item = runtime.store.get(("users",),runtime.context.user_id)
    return str(item.value) if item else "No user memory found"



# Initialize LLM

In [54]:
load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b"
)

# Run Agent

In [63]:

with (
    PostgresSaver.from_conn_string(DB_URI) as checkpointer, 
    PostgresStore.from_conn_string(DB_URI) as store):
    checkpointer.setup() # auto create tables in PostgreSQL
    store.setup() # auto create tables in PostgreSQL
    agent = create_agent(
        model=model,
        tools=[save_user_memory,get_user_memory],
        checkpointer=checkpointer,
        store=store,
        context_schema=UserContext,
        system_prompt="You are a helpful assistant that can remember user information and retrieve it when asked."
    )

    response = agent.invoke(
        {"messages":[{"role":"user","content":"What language i use"}]},
        config={"configurable":{"thread_id": "11"}},
        context=UserContext(user_id="user_101")
    )

    print(response["messages"][-1].content)

You’re using **Python**.
